<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/21-responsible-and-production-ai/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 21: Exercises

Twelve exercises: four Easy, five Medium and three Hard. They're the small tools that responsible and production machine learning is made of, and they all run in seconds with NumPy and pandas. The Easy ones measure and protect: selection rates, the disparate impact ratio, a differentially private count, and k-anonymity. The Medium ones audit and guard a deployed model: error rates per group, a threshold for a target true positive rate, the population stability index, reweighing for fairness, and validating a request before it reaches the model. The Hard ones explain and monitor: permutation importance, exact Shapley values, and the Kolmogorov-Smirnov statistic.

**Run the setup cell first.**

Rules:

- Python, NumPy and pandas. Don't call a library that does the whole job for you (such as scikit-learn's `permutation_importance`, the `shap` library or `scipy.stats.ks_2samp`); the point is to see how each piece works.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import itertools
import json
import math
import string

import numpy as np
import pandas as pd


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, np.ndarray) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_selection_rates(f):
    rng = np.random.default_rng(0)
    pred = rng.integers(0, 2, 300)
    groups = rng.choice(["x", "y", "z"], 300)
    _scenarios("selection_rates", [
        ("the example", lambda: f([1, 0, 1, 1], ["a", "a", "b", "b"]), {"a": 0.5, "b": 1.0}, None),
        ("three groups", lambda: f(pred, groups), {g: float(pred[groups == g].mean()) for g in ["x", "y", "z"]}, None),
        ("a pandas Series", lambda: f(pd.Series([0, 1, 1]), pd.Series(["m", "f", "f"])), {"m": 0.0, "f": 1.0}, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f(pred, groups).values()}), ["float"], None),
        ("different lengths raise", lambda: f([1, 0, 1], ["a", "b"]), Raises(ValueError), None),
    ])


def check_disparate_impact(f):
    pred = np.array([1, 0, 0, 0, 0, 1, 1, 0, 1, 0])
    groups = np.array(["a"] * 5 + ["b"] * 4 + ["c"])
    _scenarios("disparate_impact", [
        ("the example", lambda: f([1, 0, 0, 0, 0, 1, 1, 0, 0, 1], ["a"] * 5 + ["b"] * 4 + ["b"], "b"), {"a": 0.2 / 0.6, "b": 1.0}, None),
        ("three groups", lambda: f(pred, groups, "b"), {"a": 0.2 / 0.75, "b": 1.0, "c": 0.0}, "divide each group's rate by the reference group's"),
        ("a reference with the lower rate", lambda: f(pred, groups, "a"), {"a": 1.0, "b": 0.75 / 0.2, "c": 0.0}, None),
        ("a reference rate of 0 raises", lambda: f(pred, groups, "c"), Raises(ValueError), None),
        ("an unknown reference raises", lambda: f(pred, groups, "d"), Raises(ValueError), None),
    ])


def check_laplace_count(f):
    def reference(count, eps, seed):
        return float(count + np.random.default_rng(seed).laplace(0, 1 / eps))

    def spread(eps):
        rng = np.random.default_rng(5)
        return float(np.std([f(100, eps, rng) for _ in range(4000)]))

    _scenarios("laplace_count", [
        ("epsilon 1", lambda: f(1179, 1.0, np.random.default_rng(0)), reference(1179, 1.0, 0), "rng.laplace(0, 1 / epsilon)"),
        ("epsilon 0.1", lambda: f(1179, 0.1, np.random.default_rng(1)), reference(1179, 0.1, 1), None),
        ("smaller epsilon, more noise", lambda: spread(0.1) > 5 * spread(1.0), True, None),
        ("not rounded", lambda: type(f(10, 0.5, np.random.default_rng(2))).__name__, "float", None),
        ("epsilon 0 raises", lambda: f(10, 0, np.random.default_rng(3)), Raises(ValueError), None),
    ])


def check_k_anonymity(f):
    df = pd.DataFrame({"age": [30, 30, 30, 40, 40, 50, 50, 50, 50],
                       "zip": ["A", "A", "A", "B", "B", "C", "C", "C", "C"],
                       "country": ["x", "x", "x", "y", "y", None, "z", "z", "z"]})
    _scenarios("k_anonymity", [
        ("one column", lambda: f(df, ["age"]), 2, None),
        ("two columns", lambda: f(df, ["age", "zip"]), 2, None),
        ("missing values form their own group", lambda: f(df, ["age", "country"]), 1, "use groupby(..., dropna=False)"),
        ("a unique row", lambda: f(df.iloc[:6], ["age", "zip"]), 1, None),
        ("returns an int", lambda: type(f(df, ["zip"])).__name__, "int", None),
    ])


def check_error_rates_by_group(f):
    rng = np.random.default_rng(1)
    yt, yp = rng.integers(0, 2, 200), rng.integers(0, 2, 200)
    g = rng.choice(["p", "q"], 200)

    def reference():
        out = {}
        for grp in ["p", "q"]:
            t, p = yt[g == grp], yp[g == grp]
            out[grp] = {"tpr": float(p[t == 1].mean()), "fpr": float(p[t == 0].mean())}
        return out

    def no_positives():
        r = f([0, 0, 1], [1, 0, 1], ["a", "a", "b"])["a"]
        return [math.isnan(r["tpr"]), r["fpr"]]

    _scenarios("error_rates_by_group", [
        ("the example", lambda: f([1, 1, 0, 0, 1, 0], [1, 0, 0, 1, 1, 0], ["a", "a", "a", "a", "b", "b"]),
         {"a": {"tpr": 0.5, "fpr": 0.5}, "b": {"tpr": 1.0, "fpr": 0.0}}, None),
        ("random labels", lambda: f(yt, yp, g), reference(), "TPR over real positives, FPR over real negatives"),
        ("a group with no positives", no_positives, [True, 0.5], "return float('nan') when a rate is undefined"),
    ])


def check_threshold_for_tpr(f):
    rng = np.random.default_rng(2)
    scores = rng.random(500)
    labels = (rng.random(500) < scores).astype(int)

    def achieved(target):
        t = f(scores, labels, target)
        tpr = (scores[labels == 1] >= t).mean()
        higher = scores[labels == 1][scores[labels == 1] > t]
        next_up = higher.min() if len(higher) else None
        tpr_next = (scores[labels == 1] >= next_up).mean() if next_up is not None else 0.0
        return [bool(tpr >= target), bool(tpr_next < target)]

    _scenarios("threshold_for_tpr", [
        ("the example, target 0.5", lambda: f([0.9, 0.7, 0.4, 0.2, 0.8, 0.1], [1, 1, 1, 1, 0, 0], 0.5), 0.7, None),
        ("the example, target 0.6", lambda: f([0.9, 0.7, 0.4, 0.2, 0.8, 0.1], [1, 1, 1, 1, 0, 0], 0.6), 0.4, "you need ceil(target * positives)"),
        ("target 1 keeps every positive", lambda: f([0.9, 0.7, 0.4, 0.2], [1, 1, 1, 1], 1.0), 0.2, None),
        ("random data: reaches the target with the highest threshold", lambda: achieved(0.73), [True, True], None),
        ("no positives raise", lambda: f([0.3, 0.4], [0, 0], 0.5), Raises(ValueError), None),
        ("a target of 0 raises", lambda: f([0.3, 0.4], [1, 0], 0.0), Raises(ValueError), None),
    ])


def check_psi(f):
    rng = np.random.default_rng(3)
    base = rng.normal(0, 1, 5000)
    shifted = rng.normal(0.5, 1, 3000)

    def reference(e, a, n_bins=10, eps=1e-4):
        edges = np.unique(np.quantile(e, np.linspace(0, 1, n_bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        pe = np.clip(np.histogram(e, edges)[0] / len(e), eps, None)
        pa = np.clip(np.histogram(a, edges)[0] / len(a), eps, None)
        return float(np.sum((pa - pe) * np.log(pa / pe)))

    ints = rng.integers(0, 4, 2000)
    _scenarios("psi", [
        ("identical samples", lambda: f(base, base), 0.0, None),
        ("a shift of half a standard deviation", lambda: f(base, shifted), reference(base, shifted), None),
        ("values beyond the training range", lambda: f(base, base + 5), reference(base, base + 5), "make the outer edges -inf and +inf"),
        ("5 bins", lambda: f(base, shifted, n_bins=5), reference(base, shifted, 5), None),
        ("many repeated values", lambda: f(ints, np.r_[ints, np.full(500, 3)]), reference(ints, np.r_[ints, np.full(500, 3)]),
         "remove duplicate edges with np.unique"),
        ("returns a float", lambda: type(f(base, shifted)).__name__, "float", None),
    ])


def check_reweighing_weights(f):
    rng = np.random.default_rng(4)
    groups = rng.choice(["a", "b"], 400, p=[0.7, 0.3])
    y = (rng.random(400) < np.where(groups == "a", 0.4, 0.15)).astype(int)

    def weighted_rates():
        w = f(groups, y)
        return [float(np.sum(w * y * (groups == g)) / np.sum(w * (groups == g))) for g in ["a", "b"]]

    _scenarios("reweighing_weights", [
        ("the example", lambda: f(["a"] * 5 + ["b"] * 5, [1, 0, 0, 0, 0, 1, 1, 1, 0, 0])[0], 0.5 * 0.4 / 0.1, None),
        ("weighted positive rates are equal across groups", lambda: weighted_rates(), [float(y.mean())] * 2,
         "P(g) * P(y) / P(g and y)"),
        ("the total weight equals the number of examples", lambda: float(f(groups, y).sum()), 400.0, None),
        ("one weight per example", lambda: f(groups, y).shape, (400,), None),
        ("already independent data gets weights of 1", lambda: f(["a", "a", "b", "b"], [0, 1, 0, 1]), np.ones(4), None),
    ])


def check_validate_request(f):
    schema = {
        "age": {"type": "number", "min": 17, "max": 100},
        "hours": {"type": "number", "min": 1},
        "gain": {"type": "number", "max": 99999, "required": False},
        "sex": {"type": "category", "allowed": ["Female", "Male"]},
        "country": {"type": "category", "allowed": ["US", "Other"], "required": False},
    }
    good = {"age": 40, "hours": 38.5, "sex": "Female"}
    _scenarios("validate_request", [
        ("a valid request", lambda: f(good, schema), [], None),
        ("with optional fields", lambda: f({**good, "gain": 0, "country": "US"}, schema), [], None),
        ("missing required field", lambda: f({"age": 40, "sex": "Male"}, schema), ["missing field: hours"], None),
        ("out of range, both limits", lambda: f({**good, "age": 300}, schema), ["age must be between 17 and 100"], None),
        ("the limits themselves are allowed", lambda: f({**good, "age": 17, "hours": 1}, schema), [], None),
        ("a type error does not stop the other checks", lambda: f({**good, "age": "forty", "hours": 0}, schema),
         ["age must be a number", "hours must be at least 1"], None),
        ("only a minimum", lambda: f({**good, "hours": 0}, schema), ["hours must be at least 1"], None),
        ("only a maximum", lambda: f({**good, "gain": 1e6}, schema), ["gain must be at most 99999"], None),
        ("not a number", lambda: f({**good, "age": "forty"}, schema), ["age must be a number"], None),
        ("a boolean is not a number", lambda: f({**good, "age": True}, schema), ["age must be a number"], "isinstance(True, int) is True"),
        ("unknown category", lambda: f({**good, "sex": "X"}, schema), ["sex has unknown value 'X'"], None),
        ("several problems, in order", lambda: f({"hours": 0, "sex": "X", "hours_per_weak": 40, "extra": 1}, schema),
         ["missing field: age", "hours must be at least 1", "sex has unknown value 'X'", "unexpected field: hours_per_weak", "unexpected field: extra"],
         "schema fields first, in schema order, then unexpected fields"),
    ])


def check_permutation_importance_scores(f):
    rng = np.random.default_rng(5)
    X = rng.normal(size=(300, 4))
    y = 3 * X[:, 0] + X[:, 2] + rng.normal(0, 0.1, 300)
    w = np.array([3.0, 0.0, 1.0, 0.0])

    def predict(A):
        return A @ w

    def neg_mse(t, p):
        return -float(np.mean((t - p) ** 2))

    def reference(seed, repeats=3):
        r = np.random.default_rng(seed)
        base = neg_mse(y, predict(X))
        out = []
        for j in range(4):
            drops = []
            for _ in range(repeats):
                S = X.copy()
                S[:, j] = X[r.permutation(len(X)), j]
                drops.append(base - neg_mse(y, predict(S)))
            out.append(np.mean(drops))
        return np.array(out)

    def unchanged():
        Xc = X.copy()
        f(predict, Xc, y, neg_mse, 2, np.random.default_rng(0))
        return Xc

    _scenarios("permutation_importance_scores", [
        ("matches the reference shuffles", lambda: f(predict, X, y, neg_mse, 3, np.random.default_rng(7)), reference(7),
         "columns outer, repeats inner, one rng.permutation per repeat"),
        ("unused columns score exactly 0", lambda: f(predict, X, y, neg_mse, 2, np.random.default_rng(1))[[1, 3]], np.zeros(2), None),
        ("the most important column first", lambda: int(np.argmax(f(predict, X, y, neg_mse, 2, np.random.default_rng(1)))), 0, None),
        ("X is not changed", unchanged, X, "shuffle a copy"),
    ])


def check_shapley_values(f):
    rng = np.random.default_rng(6)
    background = rng.normal(size=(30, 4))
    x = np.array([1.0, -2.0, 0.5, 3.0])
    w = np.array([2.0, 0.0, -1.0, 0.5])

    def linear(A):
        return A @ w + 1.0

    def interaction(A):
        return A[:, 0] * A[:, 1] + np.sin(A[:, 2])

    def symmetric(A):
        return A[:, 0] + A[:, 1] + 0 * A[:, 2]

    def efficiency():
        phi = f(interaction, x[:3], background[:, :3])
        return float(phi.sum())

    _scenarios("shapley_values", [
        ("a linear model", lambda: f(linear, x, background), w * (x - background.mean(axis=0)), "w_j * (x_j - mean of background column j)"),
        ("an unused feature gets 0", lambda: float(f(linear, x, background)[1]), 0.0, None),
        ("they add up to f(x) minus the average", efficiency,
         float(interaction(x[None, :3])[0] - interaction(background[:, :3]).mean()), None),
        ("identical roles get identical values", lambda: f(symmetric, np.array([1.0, 1.0, 5.0]), np.zeros((5, 3))), np.array([1.0, 1.0, 0.0]), None),
        ("returns one value per feature", lambda: f(interaction, x[:3], background[:, :3]).shape, (3,), None),
    ])


def check_ks_statistic(f):
    from scipy import stats
    rng = np.random.default_rng(7)
    a, b = rng.normal(0, 1, 400), rng.normal(0.3, 1.2, 250)
    ta, tb = rng.integers(0, 5, 300), rng.integers(1, 6, 200)
    _scenarios("ks_statistic", [
        ("identical samples", lambda: f(a, a), 0.0, None),
        ("two normal samples", lambda: f(a, b), float(stats.ks_2samp(a, b).statistic), None),
        ("tied integer data", lambda: f(ta, tb), float(stats.ks_2samp(ta, tb).statistic), None),
        ("no overlap", lambda: f([1, 2, 3], [10, 11]), 1.0, None),
        ("no overlap, other order", lambda: f([10, 11], [1, 2, 3]), 1.0, "evaluate the CDFs at the values of both samples, and take the absolute gap"),
        ("different sizes, one sample a subset", lambda: f(a[:100], a), float(stats.ks_2samp(a[:100], a).statistic), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `selection_rates`

Write `selection_rates(y_pred, groups)` that returns, for each group, the share of people the model predicted positive (the quantity compared by demographic parity).

- `y_pred` holds 0s and 1s; `groups` holds a group label for each person (same length).
- Return a dictionary from each group label to its selection rate, as a Python `float`.
- Raise a `ValueError` if the two inputs have different lengths.

Example: `selection_rates([1, 0, 1, 1], ["a", "a", "b", "b"])` is `{"a": 0.5, "b": 1.0}`.

In [2]:
def selection_rates(y_pred, groups) -> dict:
    # your code here
    pass

In [3]:
check_selection_rates(selection_rates)

FAIL  selection_rates: the example
      expected: {'a': 0.5, 'b': 1.0}
      got:      None


### Exercise 2: `disparate_impact`

Write `disparate_impact(y_pred, groups, reference)` that compares each group's selection rate with a reference group's.

- For every group `g`, the ratio is `selection rate of g / selection rate of the reference group`.
- Return a dictionary from group to ratio (the reference group's ratio is 1.0).
- Raise a `ValueError` if the reference group's selection rate is 0, or if it doesn't appear in `groups`.

The "four-fifths rule" used in US employment guidance treats a ratio below 0.8 as a sign of possible adverse impact.

Example: with selection rates `{"a": 0.2, "b": 0.5}` and reference `"b"`, the ratios are `{"a": 0.4, "b": 1.0}`.

In [4]:
def disparate_impact(y_pred, groups, reference) -> dict:
    # your code here
    pass

In [5]:
check_disparate_impact(disparate_impact)

FAIL  disparate_impact: the example
      expected: {'a': 0.33333333333333337, 'b': 1.0}
      got:      None


### Exercise 3: `laplace_count`

Write `laplace_count(true_count, epsilon, rng)` that releases a count with differential privacy, using the Laplace mechanism from Section 9 of the lesson.

- Add noise drawn from a Laplace distribution with mean 0 and scale `1 / epsilon`, using exactly one call to `rng.laplace(0, 1 / epsilon)`.
- Return the noisy count as a `float` (don't round it).
- Raise a `ValueError` if `epsilon` isn't positive.

`rng` is a NumPy random generator such as `np.random.default_rng(0)`; the check gives your function and its own reference the same generator state.

In [6]:
def laplace_count(true_count: int, epsilon: float, rng) -> float:
    # your code here
    pass

In [7]:
check_laplace_count(laplace_count)

FAIL  laplace_count: epsilon 1
      expected: 1179.320099725158
      got:      None
      hint: rng.laplace(0, 1 / epsilon)


### Exercise 4: `k_anonymity`

Write `k_anonymity(df, quasi_identifiers)` that returns the `k` of a table: the size of the smallest group of rows that share the same values in the given columns.

- `df` is a pandas DataFrame; `quasi_identifiers` is a list of column names.
- Missing values count as a value of their own (two rows with a missing country and the same other values are in the same group).
- Return a Python `int`.

Example: if every combination of age and postcode appears at least 3 times, and one appears exactly 3 times, the table is 3-anonymous and the answer is 3.

In [8]:
def k_anonymity(df: pd.DataFrame, quasi_identifiers) -> int:
    # your code here
    pass

In [9]:
check_k_anonymity(k_anonymity)

FAIL  k_anonymity: one column
      expected: 2
      got:      None


## Medium

### Exercise 5: `error_rates_by_group`

Write `error_rates_by_group(y_true, y_pred, groups)` that returns each group's true positive rate and false positive rate, the quantities compared by equal opportunity and equalised odds.

- Return a dictionary from group label to `{"tpr": ..., "fpr": ...}`, with `float` values.
- TPR is the share of the group's real positives predicted positive; FPR is the share of its real negatives predicted positive.
- If a group has no real positives, its TPR is `float("nan")`; likewise FPR with no real negatives.

Example: for `y_true = [1, 1, 0, 0, 1, 0]`, `y_pred = [1, 0, 0, 1, 1, 0]` and `groups = ["a", "a", "a", "a", "b", "b"]`, group `"a"` has TPR 0.5 and FPR 0.5, and group `"b"` has TPR 1.0 and FPR 0.0.

In [10]:
def error_rates_by_group(y_true, y_pred, groups) -> dict:
    # your code here
    pass

In [11]:
check_error_rates_by_group(error_rates_by_group)

FAIL  error_rates_by_group: the example
      expected: {'a': {'tpr': 0.5, 'fpr': 0.5}, 'b': {'tpr': 1.0, 'fpr': 0.0}}
      got:      None


### Exercise 6: `threshold_for_tpr`

Write `threshold_for_tpr(scores, y_true, target)` that finds the decision threshold used for per-group thresholds in Section 6 of the lesson.

- Predictions are `scores >= threshold`.
- Return the **highest** threshold whose true positive rate is at least `target`. That threshold is always one of the positive examples' scores.
- Raise a `ValueError` if there are no positives, or if `target` isn't between 0 (exclusive) and 1 (inclusive).

Example: for positive scores `[0.9, 0.7, 0.4, 0.2]` (any negatives don't matter), a target of 0.5 needs 2 of the 4 positives, so the threshold is `0.7`; a target of 0.6 needs 3, so it's `0.4`.

In [12]:
def threshold_for_tpr(scores, y_true, target: float) -> float:
    # your code here
    pass

In [13]:
check_threshold_for_tpr(threshold_for_tpr)

FAIL  threshold_for_tpr: the example, target 0.5
      expected: 0.7
      got:      None


### Exercise 7: `psi`

Write `psi(expected, actual, n_bins=10, epsilon=1e-4)`, the population stability index used for drift monitoring in Section 13 of the lesson.

1. Bin edges are the quantiles of `expected` at `0, 1/n_bins, 2/n_bins, ..., 1`, with duplicate edges removed (`np.unique`). Then replace the first edge with `-inf` and the last with `+inf`, so every value of either sample falls in a bin.
2. Count each sample into the bins (`np.histogram`) and divide by the sample size to get shares `e` and `a`.
3. Replace any share below `epsilon` by `epsilon` (so empty bins don't give `log(0)`).
4. Return `sum((a - e) * log(a / e))` as a `float`.

Identical samples give 0; the larger the shift, the larger the PSI.

In [14]:
def psi(expected, actual, n_bins: int = 10, epsilon: float = 1e-4) -> float:
    # your code here
    pass

In [15]:
check_psi(psi)

FAIL  psi: identical samples
      expected: 0.0
      got:      None


### Exercise 8: `reweighing_weights`

Write `reweighing_weights(groups, y)`, the **reweighing** method of Kamiran and Calders (2012), a way to reduce bias before training.

Each training example gets a weight so that, in the weighted data, group and label are independent:

`weight(g, label) = P(group = g) * P(y = label) / P(group = g and y = label)`

where the probabilities are the shares in the data. Return a NumPy array with one weight per example. (A group-label combination that never occurs gets no weight, since no example has it.)

Example: if group `"a"` is half the data, positives are 40% of the data, and positives in group `"a"` are 10% of the data, each positive in group `"a"` gets weight `0.5 * 0.4 / 0.1 = 2.0`.

In [16]:
def reweighing_weights(groups, y) -> np.ndarray:
    # your code here
    pass

In [17]:
check_reweighing_weights(reweighing_weights)

FAIL  reweighing_weights: the example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: 2.0


### Exercise 9: `validate_request`

Write `validate_request(payload, schema)` that checks an incoming prediction request before it reaches a model, and returns a list of error messages (an empty list if it's valid).

`schema` maps each field name to a dictionary with:

- `"type"`: `"number"` or `"category"`
- for numbers, optional `"min"` and `"max"`
- for categories, `"allowed"`: a list of allowed values
- optional `"required"` (default `True`)

Rules, checked field by field in the order of `schema`, then for extra fields:

- a required field that's missing: `"missing field: NAME"`
- a number that isn't an `int` or `float` (booleans don't count as numbers): `"NAME must be a number"`
- a number outside its range: `"NAME must be between MIN and MAX"` (use the schema's values as written; if only one limit is given, say `"at least MIN"` or `"at most MAX"`)
- a category not in `"allowed"`: `"NAME has unknown value VALUE"` (with `VALUE` written as `repr(value)`)
- a field not in the schema: `"unexpected field: NAME"`, after all the other messages, in the order the payload lists them

Limits are inclusive: with `"min": 17`, the value 17 is fine. Report every problem, not just the first one, so that a caller can fix them all in one go.

In [18]:
def validate_request(payload: dict, schema: dict) -> list:
    # your code here
    pass

In [19]:
check_validate_request(validate_request)

FAIL  validate_request: a valid request
      expected: []
      got:      None


## Hard

### Exercise 10: `permutation_importance_scores`

Write `permutation_importance_scores(predict, X, y, metric, n_repeats, rng)` that measures how much a model relies on each column, as in Section 7 of the lesson.

- `predict` takes a 2D NumPy array and returns predictions; `X` is a 2D NumPy array; `metric(y, predictions)` returns a score where higher is better.
- First compute the baseline score on the unshuffled data.
- Then, **for each column in order, and for each repeat**, shuffle that one column with `rng.permutation(len(X))` (one call per repeat), compute the score, and record `baseline - shuffled score`.
- Return a NumPy array with the **mean** drop for each column.
- Don't change `X`.

Use the random generator exactly in that order (columns outer, repeats inner), so the check can reproduce your shuffles.

In [20]:
def permutation_importance_scores(predict, X: np.ndarray, y, metric, n_repeats: int, rng) -> np.ndarray:
    # your code here
    pass

In [21]:
check_permutation_importance_scores(permutation_importance_scores)

FAIL  permutation_importance_scores: matches the reference shuffles
      expected: array(shape=(4,), dtype=float64, [16.5182, 0. , 1.9182, 0. ])
      got:      None
      hint: columns outer, repeats inner, one rng.permutation per repeat


### Exercise 11: `shapley_values`

Write `shapley_values(f, x, background)` that computes exact interventional Shapley values for one example, as in Section 8 of the lesson.

- `f` takes a 2D NumPy array of rows and returns one number per row.
- `x` is a 1D array (the example, with `d` features); `background` is a 2D array of reference rows.
- For a subset `S` of features, `v(S)` is the mean of `f` over the background rows, with the features in `S` replaced by `x`'s values.
- Feature `j`'s Shapley value is the sum over all subsets `S` not containing `j` of `|S|! (d - |S| - 1)! / d! * (v(S with j) - v(S))`.
- Return a NumPy array of `d` values.

Their sum equals `f(x) - mean(f(background))` (the check tests this).

In [22]:
def shapley_values(f, x: np.ndarray, background: np.ndarray) -> np.ndarray:
    # your code here
    pass

In [23]:
check_shapley_values(shapley_values)

FAIL  shapley_values: a linear model
      expected: array(shape=(4,), dtype=float64, [ 2.6221, -0. , -0.6671, 1.4474])
      got:      None
      hint: w_j * (x_j - mean of background column j)


### Exercise 12: `ks_statistic`

Write `ks_statistic(a, b)` that returns the two-sample Kolmogorov-Smirnov statistic, used in Section 13 of the lesson to test whether a column has drifted.

- The statistic is the largest absolute difference between the two samples' **empirical cumulative distribution functions**: for any value `t`, `F_a(t)` is the share of `a` that is `<= t`, and likewise `F_b(t)`.
- The largest difference always occurs at one of the observed values, so it's enough to compare `F_a` and `F_b` at every value in either sample.
- Handle ties correctly (both samples can contain repeated values).
- Return a `float`. Don't use `scipy`; the check compares your answer with `scipy.stats.ks_2samp`.

In [24]:
def ks_statistic(a, b) -> float:
    # your code here
    pass

In [25]:
check_ks_statistic(ks_statistic)

FAIL  ks_statistic: identical samples
      expected: 0.0
      got:      None


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the capstone in `04_mini_project.ipynb`, where you'll take one model all the way from raw data to a monitored, documented service.